In [ ]:
# Lab type: debug
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: Backtesting Without Temporal Leakage
# Task: The backtest harness below contains 3 bugs — the three evaluation leaks
#       from the lesson. Each runs cleanly and inflates the reported score.
#       Find each bug, explain it, and fix it in the fix cell.

# Lab: Debugging a Backtest Harness

A teammate wrote a cross-validated evaluation for the 7-day-ahead orders model and
reports it "beats the baseline comfortably." The harness runs without error. All
three bugs are in the *evaluation*, not the model.

**Outputs are cleared.** Run each cell to generate results.

## Setup: features for a 7-day-ahead model

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit

# Deployment: each day we forecast 7 days ahead → features must be shifted ≥ 7
df = pd.DataFrame({"orders": orders})
df["dayofweek"] = df.index.dayofweek
df["month"] = df.index.month
df["lag_7"] = df["orders"].shift(7)
df["lag_14"] = df["orders"].shift(14)
df["roll_7"] = df["orders"].shift(7).rolling(7).mean()
df = df.dropna()

FEATS = ["dayofweek", "month", "lag_7", "lag_14", "roll_7"]
X, y = df[FEATS], df["orders"]
print(f"{len(df)} usable rows")

## Bug 1: The convenient helper

In [ ]:
# --- BUGGY CODE (Bug 1) ---
# Review this evaluation — what folds does cv=5 actually build?
model = HistGradientBoostingRegressor(random_state=0)
scores = cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
print(f"cross_val_score MAE per fold: {(-scores).round(1)}")
print(f"mean MAE: {-scores.mean():.1f}")

**Explain the bug:** `cv=5` didn't shuffle anything — the folds are contiguous blocks.
So why is this still temporal leakage? Think about which side of each test block the
training data sits on, especially for the first fold.

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**The bug:** K-fold rotates the test block through the series: fold 1 tests on the
*earliest* fifth while training on everything after it — the model literally trains on
2024–2025 to predict 2023. Every fold except the last trains partly on the future of
its test window. No shuffling required; the fold *design* leaks.

**Correct approach:** `cross_val_score(model, X, y, cv=TimeSeriesSplit(n_splits=5, test_size=90), ...)`
— every test window lies strictly after its training window.

</details>

In [ ]:
# Fix for Bug 1: chronological folds
tscv = TimeSeriesSplit(n_splits=5, test_size=90)
scores_fix = cross_val_score(model, X, y, cv=tscv, scoring="neg_mean_absolute_error")
print(f"TimeSeriesSplit MAE per fold: {(-scores_fix).round(1)}")
print(f"mean MAE: {-scores_fix.mean():.1f}  (higher than Bug 1 — that gap was leakage)")

## Bug 2: Preprocessing outside the folds

In [ ]:
# --- BUGGY CODE (Bug 2) ---
# Review this pipeline — the folds are now chronological. Is the evaluation clean?
X_scaled = StandardScaler().fit_transform(X)      # ← fit once, on everything

ridge_scores = cross_val_score(Ridge(), X_scaled, y, cv=tscv,
                               scoring="neg_mean_absolute_error")
print(f"Ridge (pre-scaled) MAE per fold: {(-ridge_scores).round(1)}")

**Explain the bug:** The folds respect time order now. Explain why fold 1's evaluation
is still contaminated, and why passing an unscaled `X` with a `Pipeline` is not just
style — what does it *structurally* guarantee that this code can't?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**The bug:** The scaler was fit once on all rows, so fold 1's training features were
standardised using means/variances from years that fold hasn't reached yet — on a
trending series, statistics that encode the future's level. Chronological folds can't
undo preprocessing that already saw everything.

**The structural fix:** `cross_val_score(Pipeline([...]), X, y, cv=tscv)` refits the
scaler *inside each fold, on that fold's training window only* — the correct fit
boundary stops being a discipline you remember and becomes something the code
enforces.

</details>

In [ ]:
# Fix for Bug 2: scaler inside the pipeline, refit per fold
pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge())])
ridge_fix = cross_val_score(pipe, X, y, cv=tscv, scoring="neg_mean_absolute_error")
print(f"Ridge (per-fold scaling) MAE per fold: {(-ridge_fix).round(1)}")

## Bug 3: The missing gap

In [ ]:
# --- BUGGY CODE (Bug 3) ---
# Review the fold design — remember the deployment: forecasts are made 7 days ahead.
tscv_nogap = TimeSeriesSplit(n_splits=5, test_size=90)   # ← gap defaults to 0

for fold, (tr_idx, te_idx) in enumerate(tscv_nogap.split(X), 1):
    train_end = df.index[tr_idx[-1]].date()
    test_start = df.index[te_idx[0]].date()
    print(f"fold {fold}: train ends {train_end} → test starts {test_start}")

**Explain the bug:** Test windows start the day after training ends. Our features are
shifted by 7 (`lag_7`, `shift(7).rolling(7)`), and the deployed forecast is made 7 days
in advance. For the first 6 rows of each test window, what information do the features
contain that the deployed model would not have — and which TimeSeriesSplit parameter
closes the hole?

*(Write your diagnosis here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**The bug:** A forecast for test day *d* is issued on day *d−7*. For the first 6 test
rows, days *d−7 … train_end* include days *inside the training window* — that's fine —
but the issue is subtler: the model's fit itself used training rows whose *targets*
run right up to the day before the test window. Deployed on day *d−7*, the newest
target the fitting process could have seen is *d−7*, not *d−1*. The evaluation
therefore hands the model 6 days of extra training history it wouldn't have — an
optimistic bias at every fold boundary.

**Correct approach:** `TimeSeriesSplit(n_splits=5, test_size=90, gap=7)` inserts a
7-day embargo between train and test, aligning "last training target" with what the
deployment timeline actually permits. Rule: **gap ≥ forecast horizon** (extend it
further if data arrives late in production).

</details>

In [ ]:
# Fix for Bug 3: embargo matching the 7-day horizon
tscv_gap = TimeSeriesSplit(n_splits=5, test_size=90, gap=7)
scores_gap = cross_val_score(model, X, y, cv=tscv_gap, scoring="neg_mean_absolute_error")
print(f"gapped folds MAE: {(-scores_gap).round(1)}   mean {-scores_gap.mean():.1f}")

for fold, (tr_idx, te_idx) in enumerate(tscv_gap.split(X), 1):
    print(f"fold {fold}: train ends {df.index[tr_idx[-1]].date()} → "
          f"test starts {df.index[te_idx[0]].date()}  (7-day embargo)")

## Summary

> **For each bug, complete the sentence in one line.**

1. **K-fold:** Unshuffled K-fold still leaks because every fold except the last trains on ________.
2. **Scaler:** Chronological folds can't fix preprocessing that was ________.
3. **Gap:** The embargo between train and test must be at least ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. ...trains on **data from the future of its own test window**.
2. ...preprocessing that was **fit on the full series before the folds were built**.
3. ...at least **the forecast horizon the features and deployment assume** (longer if
   production data arrives with a delay).

</details>